# 🧷 36 · Regulatory genomics: deep learning from DNA sequence to function
### Motifs as position weight matrices, k-mer baselines, a convolutional network trained with chromosome hold-out, filters and attributions, cis-regulatory grammar by in-silico experiments, non-coding variant-effect prediction and genomic language models

**BiologyToML** · Level 2 · Research frontier · Part H — Genomes, proteins and molecules · Module 36 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/36_regulatory_genomics_deep_learning.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, Matplotlib, PyTorch (CPU) |
| **Prerequisites** | Level 1, especially Modules 07 (ridge), 19 (HMMs: position-specific emissions, Markov chains), 22 (neural networks and a 1-D CNN on DNA in PyTorch) and 23 (held-out evaluation); Module 24 (attention) is helpful for Section 8 |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **Builds on** | [07 · Ridge, lasso and sparse genetic effects](07_ridge_lasso_sparse_regression.ipynb), [10 · Decision trees and random forests](10_decision_trees_random_forests.ipynb), [19 · Hidden Markov models I](19_hmm_inference.ipynb), [22 · Neural networks](22_neural_networks.ipynb), [23 · Model selection](23_model_selection.ipynb), [24 · Attention and transformers](24_attention_transformers.ipynb), [35 · Statistical genetics at scale](35_statistical_genetics.ipynb) |
| **Research frontier covered** | sequence-to-function models (DeepSEA, Basset, BPNet, Enformer), motif discovery and attribution, cis-regulatory grammar, non-coding variant effect prediction, genomic language models — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED** with `simulate_regulatory_sequences`, a stand-in for a massively parallel reporter
> assay (MPRA/STARR-seq): 6,000 random 200-bp sequences with planted transcription-factor motifs (AP-1, GATA, SP1,
> E-box and the repressor REST), each assigned to one of 8 pretend chromosomes, and a measured "enhancer activity"
> = a hidden motif grammar + noise. The grammar is implemented in the toolkit function `regulatory_activity` — the
> **answer key**, which you should not read until Section 6. Real regulatory DNA has thousands of factors, chromatin,
> cell-type specificity and long-range interactions; Sections 8–9 explain what changes.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** how transcription factors, motifs and enhancers control gene expression and what ChIP-seq, ATAC-seq and massively parallel reporter assays measure — including their typical artefacts.
2. **Build** a position weight matrix from aligned binding sites, **compute** its information content and **scan** sequences on both strands with log-odds scores.
3. **Set up** an honest benchmark — chromosome hold-out, a noise ceiling, an oracle-feature reference and a k-mer ridge baseline — before training any neural network.
4. **Implement** and **train** a convolutional network for sequence → activity in PyTorch (with reverse-complement augmentation) and **compare** it with the baselines across training-set sizes.
5. **Open the box**: turn first-layer filters into PWMs and **match** them to the true motifs; **compute** in-silico mutagenesis and gradient × input attributions and **score** them against the planted sites.
6. **Probe** the learned cis-regulatory grammar (spacing, orientation, position) with in-silico experiments and **compare** it with the ground-truth grammar.
7. **Predict** the effect of every single-nucleotide variant, **evaluate** against the true effects, and **contrast** supervised predictions with zero-shot scores from a (Markov-chain) DNA language model.
8. **Describe** long-context models (Basenji, Enformer, Borzoi, AlphaGenome) and genomic language models (DNABERT, Nucleotide Transformer, Evo), and **state** their documented limitations.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · Gene regulation and the assays | TFs, enhancers, ChIP/ATAC/MPRA; the simulated reporter assay | 15 min |
| 2 · Motifs, PWMs and baselines | PWM, information content, scanning; noise ceiling, oracle features, k-mer ridge | 25 min |
| 3 · A CNN for sequence → activity | architecture, reverse-complement augmentation, chromosome hold-out, learning curve | 25 min |
| 4 · Opening the box: filters | activation-based PWMs, matching to the true motifs | 15 min |
| 5 · Attribution | in-silico mutagenesis, gradient × input, AUROC against planted sites | 20 min |
| 6 · Grammar by in-silico experiments | spacing, orientation and position vs the answer key | 20 min |
| 7 · Variant-effect prediction | all 36,000 SNVs of 60 test sequences vs true effects | 15 min |
| 8 · Large models and genomic language models | context length; a Markov-chain DNA LM for zero-shot scoring | 15 min |
| 9 · 🧭 From toy to real data | ENCODE, JASPAR, TF-MoDISco, what goes wrong | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 15 min |

## 🧫 The biological question

Every cell in your body carries the same genome, yet a liver cell and a neuron express very different genes. Much of
the difference is written in **regulatory DNA**: short stretches — enhancers, promoters, silencers — that recruit
**transcription factors** (TFs). Each TF recognises a short, fuzzy DNA word (a **motif**, 6–20 bp); an enhancer
switches its gene on when the right combination of factors binds in the right arrangement — a **cis-regulatory
grammar** that we still only partly understand.

This matters for medicine. Roughly nine in ten variants found by genome-wide association studies lie outside
protein-coding sequence, and they are concentrated in regulatory DNA marked by open chromatin (Maurano *et al.* 2012).
Fine-mapping ([Module 35](35_statistical_genetics.ipynb)) narrows a locus to a credible set of such variants — but which
one changes regulation, in which cell type, and in which direction?

Since 2015, **sequence-to-function** deep-learning models have attacked this question: DeepBind and DeepSEA learned
motif-like filters from ChIP-seq and chromatin data; Basset, Basenji, BPNet and Enformer followed with more data and
longer context; DeepSTARR predicted enhancer activity measured by a genome-wide reporter assay in fly cells and designed
synthetic enhancers (de Almeida *et al.* 2022). The recipe is always the same: **learn $f(\text{DNA}) \to$ measurement,
then interrogate $f$** — open it to find motifs and grammar, and ask it what happens when a single letter changes.

We follow that recipe end-to-end on a simulated **reporter assay** — 6,000 sequences of 200 bp whose activity is
determined by a hidden grammar of five real TF motifs, measured with noise. Because we wrote the grammar, we can check
whether the network learned the motifs, their grammar and the effects of mutations. The simulation mimics the design
of MPRA/STARR-seq experiments (Melnikov *et al.* 2012; Arnold *et al.* 2013) used by models such as DeepSTARR, in a
version small enough to train on a laptop CPU in well under a minute.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration ──
torch.manual_seed(SEED)                                   # PyTorch has its own random-number generator
torch.set_num_threads(min(2, torch.get_num_threads()))    # ≤ 2 threads: plenty for small CNNs, polite on shared machines
print("PyTorch", torch.__version__, "| CPU threads:", torch.get_num_threads())

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use the reporter-assay simulator with its motif table (IUPAC consensus and effect of each motif), motif search on both strands, reverse complements and one-hot encoding. ⚠️ The copy includes `regulatory_activity`, the hidden grammar: please do **not** read its docstring or code before Section 6 — discovering it is the point. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd

IUPAC = {"A": "A", "C": "C", "G": "G", "T": "T", "R": "AG", "Y": "CT", "S": "CG", "W": "AT", "K": "GT",
         "M": "AC", "N": "ACGT"}


REGULATORY_MOTIFS = {
    "AP-1": ("TGASTCA", 1.0),
    "GATA": ("WGATAR", 0.7),
    "SP1": ("GGGGCGGGG", 0.5),
    "E-box": ("CACGTG", 0.8),
    "REST": ("TTCAGCACC", -1.5),        # repressor
}


def reverse_complement(seq: str) -> str:
    """Reverse complement of a DNA string (IUPAC codes are complemented too)."""
    comp = {"A": "T", "C": "G", "G": "C", "T": "A", "R": "Y", "Y": "R", "S": "S", "W": "W", "K": "M",
            "M": "K", "N": "N"}
    return "".join(comp[b] for b in reversed(seq.upper()))


def find_motif_sites(seq: str, consensus: str) -> list:
    """Start positions where ``consensus`` (IUPAC) matches ``seq`` on either strand (overlaps allowed)."""
    import re
    sites = set()
    for c in {consensus, reverse_complement(consensus)}:
        pattern = "(?=" + "".join(f"[{IUPAC[b]}]" for b in c) + ")"
        sites.update(m.start() for m in re.finditer(pattern, seq))
    return sorted(sites)


def regulatory_activity(seq: str) -> float:
    """Noise-free enhancer activity of a DNA sequence under the course's motif grammar.

    Activity = Σ motif effects (sites in the central half count fully, flanking sites 60 %)
    + 1.0 if an AP-1 and a GATA site lie within 30 bp of each other (cooperative binding)
    + 2·(GC fraction − 0.5).  This is the "true biology" that Module 36's networks must learn.
    """
    L = len(seq)
    total, starts = 0.0, {}
    for name, (consensus, effect) in REGULATORY_MOTIFS.items():
        starts[name] = find_motif_sites(seq, consensus)
        for s in starts[name]:
            total += effect * (1.0 if L / 4 <= s + len(consensus) / 2 <= 3 * L / 4 else 0.6)
    if any(abs(a - g) <= 30 for a in starts["AP-1"] for g in starts["GATA"]):
        total += 1.0
    gc = (seq.count("G") + seq.count("C")) / max(L, 1)
    return total + 2.0 * (gc - 0.5)


def simulate_regulatory_sequences(rng: np.random.Generator, n_seqs: int = 4000, length: int = 200,
                                  motif_rate: float = 1.5, noise: float = 0.3, n_chromosomes: int = 8) -> dict:
    """Simulate an enhancer reporter assay (MPRA / STARR-seq style): DNA sequence → activity.

    Background DNA has a per-sequence GC content; each sequence receives Poisson(``motif_rate``)
    planted motif instances at random positions and orientations (sampled from the IUPAC
    consensus).  The measured activity is ``regulatory_activity(seq)`` plus Gaussian noise,
    so motifs that occur by chance count too, exactly as in real DNA.  Each sequence is
    assigned to a chromosome so that models can be tested on held-out chromosomes.

    Returns a dict with ``sequences`` (list of str), ``activity``, ``label`` (1 = active, top
    40 %), ``chromosome`` and ``sites`` (DataFrame of planted motifs).
    """
    names = list(REGULATORY_MOTIFS)
    seqs, rows = [], []
    for i in range(n_seqs):
        gc = rng.beta(8, 8)
        s = list(rng.choice(list("ACGT"), length, p=[(1 - gc) / 2, gc / 2, gc / 2, (1 - gc) / 2]))
        for _ in range(rng.poisson(motif_rate)):
            m = names[rng.integers(len(names))]
            inst = "".join(rng.choice(list(IUPAC[b])) for b in REGULATORY_MOTIFS[m][0])
            strand = "+" if rng.random() < 0.5 else "-"
            if strand == "-":
                inst = reverse_complement(inst)
            start = int(rng.integers(0, length - len(inst)))
            s[start:start + len(inst)] = list(inst)
            rows.append((i, m, start, strand))
        seqs.append("".join(s))
    activity = np.array([regulatory_activity(s) for s in seqs]) + rng.normal(0, noise, n_seqs)
    label = (activity > np.quantile(activity, 0.6)).astype(int)
    chromosome = np.array([f"chr{c + 1}" for c in rng.integers(0, n_chromosomes, n_seqs)], dtype=object)
    sites = pd.DataFrame(rows, columns=["seq_id", "motif", "start", "strand"])
    return dict(sequences=seqs, activity=activity, label=label, chromosome=chromosome, sites=sites)


def one_hot_dna(seq: str) -> np.ndarray:
    """One-hot encode a DNA string as an L × 4 array with columns A, C, G, T (N → all zeros)."""
    lookup = {"A": 0, "C": 1, "G": 2, "T": 3}
    out = np.zeros((len(seq), 4))
    for i, base in enumerate(seq.upper()):
        if base in lookup:
            out[i, lookup[base]] = 1.0
    return out


print("🧰 Toolkit loaded:", ", ".join(['reverse_complement', 'find_motif_sites', 'regulatory_activity', 'simulate_regulatory_sequences', 'one_hot_dna']))

---
## 1 · Gene regulation and the assays that measure it

> 📍 **Where we are.** In [Module 22](22_neural_networks.ipynb) (Section 8) a CNN found a single planted E-box in
> random DNA. Real regulatory DNA is a *combination* of motifs whose effects depend on each other and on position.
> Here we meet the biology, the assays and our simulated data set.

**The players.** A **transcription factor** has a DNA-binding domain that prefers certain base sequences; its
preference is summarised by a **motif** — a consensus like `TGASTCA` for AP-1 (in IUPAC code, `S` = C or G). A
**promoter** sits at the start of a gene; **enhancers** can lie up to hundreds of kilobases (occasionally about a
megabase) away and contact the promoter by looping of chromatin. **Activators** recruit the transcription machinery,
**repressors** (like REST, which silences neuronal genes outside neurons) recruit silencing complexes. Binding needs
accessible chromatin: DNA wrapped tightly around nucleosomes is mostly hidden from TFs.

🔬 **Assay box — reading regulation genome-wide.**
- **ChIP-seq** (chromatin immunoprecipitation + sequencing): cross-link proteins to DNA, pull down one TF (or histone
  mark) with an antibody, sequence the attached DNA. *Output:* peaks where the factor binds (~10⁴–10⁵ per experiment).
  *Artefacts:* antibody specificity, "hyper-ChIPable" open regions, peak-calling thresholds, ~200-bp resolution
  (ChIP-exo/ChIP-nexus reach near-base resolution, as used by BPNet).
- **ATAC-seq / DNase-seq**: an enzyme (Tn5 transposase or DNase I) cuts or tags *accessible* DNA. *Output:* open-chromatin
  peaks in a cell type (~10⁵); footprints of bound factors. *Artefacts:* Tn5 sequence bias, mitochondrial reads,
  GC and mappability biases.
- **MPRA / STARR-seq** (massively parallel reporter assays): clone thousands to millions of candidate sequences into a
  reporter construct, each tagged so that its transcriptional output can be counted by sequencing. In STARR-seq the
  candidate sits in the transcript and reports its own activity. *Output:* a quantitative **activity** per sequence —
  a direct measure of sequence → function. *Artefacts:* episomal context (no native chromatin), promoter and cell-line
  dependence, barcode and library-size noise.

Our simulation mimics an MPRA: every sequence gets an activity value, with measurement noise.

In [ ]:
# ── 1.1 A SIMULATED reporter assay: 6,000 sequences × 200 bp with planted motifs ──
rng = np.random.default_rng(SEED)
data = simulate_regulatory_sequences(rng, n_seqs=6000, length=200, motif_rate=3.0)
seqs, activity, chrom, sites = data["sequences"], data["activity"], data["chromosome"], data["sites"]
N_SEQ, L_SEQ = len(seqs), len(seqs[0])
print(f"SIMULATED MPRA: {N_SEQ} sequences of {L_SEQ} bp | chromosomes: "
      f"{ {c: int(n) for c, n in zip(*np.unique(chrom, return_counts=True))} }")
print("motif table (IUPAC consensus, effect):", REGULATORY_MOTIFS)
print(f"planted motif instances: {len(sites)} ({len(sites) / N_SEQ:.1f} per sequence on average)")
print(f"measured activity: mean {activity.mean():.2f}, SD {activity.std():.2f}, range {activity.min():.1f} to {activity.max():.1f}")
print("first sequence:", seqs[0][:70], "…")
sites.head(6)

In [ ]:
# ── 1.2 A first look at the data (without peeking at the answer key) ──
motifs = list(REGULATORY_MOTIFS)
planted = sites.groupby(["seq_id", "motif"]).size().unstack(fill_value=0).reindex(range(N_SEQ), fill_value=0)
gc = np.array([(s.count("G") + s.count("C")) / L_SEQ for s in seqs])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].hist(activity, bins=60, color="tab:blue")
axes[0].set_xlabel("measured activity"); axes[0].set_ylabel("sequences")
axes[0].set_title("activity distribution (SIMULATED MPRA)")
for k, m in enumerate(motifs):
    means = [activity[planted[m].values == c].mean() for c in (0, 1, 2)]
    axes[1].plot([0, 1, 2], means, "o-", color=plt.cm.tab10(k), label=m)
axes[1].set_xticks([0, 1, 2]); axes[1].set_xlabel("planted copies of the motif"); axes[1].set_ylabel("mean activity")
axes[1].set_title("activators raise, REST lowers activity"); axes[1].legend(fontsize=7)
axes[2].scatter(gc, activity, s=2, alpha=0.4, color="0.4")
axes[2].set_xlabel("GC fraction of the sequence"); axes[2].set_ylabel("measured activity")
axes[2].set_title(f"GC content: r = {np.corrcoef(gc, activity)[0, 1]:.2f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each sequence carries on average 3.0 planted motif instances, at random positions and on
random strands, in background DNA whose GC content varies from sequence to sequence (a "motif-rich" library; the
simulator's default is 1.5 per sequence). Activity rises with the number of planted AP-1, GATA, E-box and SP1 copies and
falls with REST copies; GC content is only weakly correlated with activity on its own (r = 0.06). Measured activity has
mean 1.05 and standard deviation 1.53 and spans −6.2 to 7.6.

These plots use the *planted* sites — information no real experiment has. A real analysis starts from sequences and
measurements only and has to *discover* the motifs, which is what the rest of the notebook does. We will use the planted
sites only to **check** what the models found.

⚠️ **Do not peek.** The toolkit cell contains `regulatory_activity`, the exact rule that turns a sequence into its
noise-free activity. Treat it as the sealed answer key of a lab experiment; we open it in Section 6.

✏️ **Your turn.** Motifs are planted one after the other at random positions, so a later motif can overwrite part of
an earlier one. Using `sites`, count for each sequence the pairs of planted motifs whose start positions are closer
than 9 bp (the longest motif). Roughly what fraction of planted sites could have been damaged? (Keep the number in mind
for the logo in Section 2.)

---
## 2 · Motifs as position weight matrices — and honest baselines

> 📍 **Where we are.** Module 22 showed that a convolution filter scores DNA exactly like a **position weight matrix**
> (PWM); Module 19 used position-specific emission probabilities in HMMs. Here we build a PWM from data, measure how
> informative it is, and use classical features to set the bar that any deep model must clear.

🧮 **Math refresher — PWMs, log-odds and information content.** Align $n$ binding sites of length $K$ and count base $c$
at position $u$: $n_{u,c}$. The **position probability matrix** is $P_{u,c}=(n_{u,c}+\alpha)/(n+4\alpha)$ with a small
pseudocount $\alpha$ (so that unseen bases do not get probability zero). Against a uniform background ($0.25$ per base)
a window $s_1\dots s_K$ is scored by the **log-odds**

$$S(s)=\sum_{u=1}^{K}\log_2\frac{P_{u,s_u}}{0.25}\quad\text{(bits)},\qquad
\mathrm{IC}_u = 2+\sum_{c}P_{u,c}\log_2P_{u,c}\quad\text{(bits)} .$$

*In words:* $S$ adds, position by position, how much more likely the observed base is under the motif than under
random DNA (Durbin *et al.* 1998) — the log-likelihood ratio of "bound site" vs "background" ([Module 05](05_likelihood_mle_map.ipynb)), assuming independent positions.
$\mathrm{IC}_u$ (information content) is 2 bits minus the entropy of column $u$: 2 bits for a perfectly conserved base,
1 bit for a two-way choice such as `S` (C or G), 0 for a column that could be anything. A **sequence logo**
(Schneider & Stephens 1990) draws each column as a stack of letters with total height $\mathrm{IC}_u$, each letter
scaled by its probability. The reverse complement of a PWM is the matrix with positions reversed and bases complemented.

> 🔗 **Builds on — [Module 22](22_neural_networks.ipynb), Section 8** (a convolution filter with log-odds weights *is* a
> PWM score) and **[Module 19](19_hmm_inference.ipynb)** (position-specific emission probabilities: a PWM is the emission
> table of an HMM path that walks through the motif one position at a time, and its log-odds score is the
> log-likelihood ratio of "motif state path" vs "background state").

In [ ]:
# ── 2.1 Helpers: aligned planted sites, position probability matrices and sequence logos ──
BASES = "ACGT"
BASE_COLORS = {"A": "tab:green", "C": "tab:blue", "G": "tab:orange", "T": "tab:red"}


def aligned_sites(motif):
    """Sequences of all planted instances of `motif`, written on the motif's own strand."""
    K = len(REGULATORY_MOTIFS[motif][0])
    out = []
    for i, start, strand in sites.loc[sites.motif == motif, ["seq_id", "start", "strand"]].itertuples(index=False):
        sub = seqs[i][start:start + K]
        out.append(reverse_complement(sub) if strand == "-" else sub)
    return out


def pwm_from_sites(site_list, pseudo=0.5):
    """Position probability matrix (K × 4, columns A C G T) with a pseudocount."""
    counts = np.array([[sum(s[u] == b for s in site_list) for b in BASES] for u in range(len(site_list[0]))])
    return (counts + pseudo) / (counts + pseudo).sum(axis=1, keepdims=True)


def draw_logo(ax, P, title=""):
    """Sequence logo with stacked bars: column height = information content, letters scaled by probability."""
    ic = 2 + np.sum(P * np.log2(P), axis=1)
    for u in range(len(P)):
        bottom = 0.0
        for c in np.argsort(P[u]):                          # small letters at the bottom, the winner on top
            h = P[u, c] * ic[u]
            ax.bar(u, h, bottom=bottom, width=0.9, color=BASE_COLORS[BASES[c]], edgecolor="white", lw=0.3)
            if h > 0.25:
                ax.text(u, bottom + h / 2, BASES[c], ha="center", va="center", fontsize=8, color="white", weight="bold")
            bottom += h
    ax.set_ylim(0, 2); ax.set_xticks(range(len(P))); ax.set_ylabel("bits"); ax.set_title(title, fontsize=9)

In [ ]:
# ── 2.2 The AP-1 PWM from the planted sites, its information content and its logo ──
ap1_sites = aligned_sites("AP-1")
P_ap1 = pwm_from_sites(ap1_sites)
W_ap1 = np.log2(P_ap1 / 0.25)                                 # log-odds weights = a convolution filter
ic_ap1 = 2 + np.sum(P_ap1 * np.log2(P_ap1), axis=1)
print(f"{len(ap1_sites)} planted AP-1 sites, e.g. {ap1_sites[:4]}")
print("information content per position (bits):", np.round(ic_ap1, 2), f"| total {ic_ap1.sum():.1f} bits")
fig, ax = plt.subplots(figsize=(4, 2.3))
draw_logo(ax, P_ap1, "AP-1 logo from planted sites (consensus TGASTCA)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.3 Scan every sequence with the AP-1 PWM on both strands ──
def pwm_scan(seq, W):
    """Log-odds score of every window on both strands; returns the best score (bits)."""
    x = one_hot_dna(seq)
    windows = np.lib.stride_tricks.sliding_window_view(x, W.shape)[:, 0]     # (L-K+1, K, 4)
    fwd = np.einsum("pkc,kc->p", windows, W)
    rev = np.einsum("pkc,kc->p", windows, W[::-1, ::-1])                       # reverse-complement PWM
    return max(fwd.max(), rev.max())


best = np.array([pwm_scan(s, W_ap1) for s in seqs])
has_ap1 = np.array([len(find_motif_sites(s, REGULATORY_MOTIFS["AP-1"][0])) > 0 for s in seqs])
print(f"maximum possible score {W_ap1.max(axis=1).sum():.1f} bits | sequences with an exact AP-1 match: {has_ap1.mean():.1%}")
print(f"best score: with a match {best[has_ap1].mean():.1f} bits on average, without {best[~has_ap1].mean():.1f} bits")
cost = W_ap1.max(axis=1, keepdims=True) - W_ap1               # bits lost when a base replaces the best one
fixed = [0, 1, 2, 4, 5, 6]                                     # column 3 is the degenerate S (C or G)
print(f"cost of one mismatch: fixed positions {np.sort(cost[fixed], axis=1)[:, 1:].min():.1f}–{cost[fixed].max():.1f} bits | "
      f"S position: A {cost[3, 0]:.1f}, T {cost[3, 3]:.1f}, C↔G {abs(W_ap1[3, 1] - W_ap1[3, 2]):.2f} bits")
bins = np.linspace(best.min(), best.max() + 0.1, 50)          # shared bins (exact matches all reach the maximum)
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.hist(best[~has_ap1], bins=bins, alpha=0.7, label="no exact AP-1 match", color="0.6")
ax.hist(best[has_ap1], bins=bins, alpha=0.7, label="contains an AP-1 match", color="tab:red")
ax.set_xlabel("best AP-1 log-odds score in the sequence (bits)"); ax.set_ylabel("sequences"); ax.legend(fontsize=7)
ax.set_title("a PWM scan separates sequences with and without the motif (SIMULATED)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The 3,633 planted AP-1 sites give a crisp logo: the six fixed positions carry about 1.7 bits
each and the middle `S` position about 0.9 bit (C or G) — 11.1 bits in total. (A perfectly clean set would give 2 bits
and 1 bit, 13 bits in total; ours is a little fuzzier because some planted sites were later partly overwritten by other
motifs planted on top of them — real binding sites are fuzzy for biological reasons.) The more bits, the rarer a chance
match: about one in $2^{13}\approx8{,}000$ random windows per strand matches the exact consensus. Scanning with the
log-odds PWM, every sequence with an exact AP-1 match reaches the maximal score (12.6 bits), while the best window of the
other sequences scores 2.5 bits on average. The clusters of the grey histogram are best windows with one or more
mismatches: one mismatch costs 5.8–6.8 bits at a fixed position and 5.3–6.0 bits if the `S` position carries A or T,
while swapping C↔G at `S` costs nothing (both are consensus) — so the cluster near 6–7 bits mixes both kinds of single
mismatch. The scan separates the two groups completely. This is how motif databases
(JASPAR, CIS-BP) are used in practice: a PWM per factor, a scan, a score threshold.

**Honest baselines.** Before training a neural network we need to know what "good" means:

- **The noise ceiling.** Measured activity = true activity + noise, so even a perfect model cannot correlate with the
  measurements better than the true activity does. Because we simulated the data we can compute it exactly; in real
  experiments it is estimated from replicate measurements.
- **An oracle-feature model (a cheating reference).** Count the exact matches of each of the five true motifs (both
  strands), add GC content, fit linear regression. It knows the motifs — which no real analysis does — but not their
  grammar.
- **A k-mer model (no prior knowledge).** Count all 6-letter words (merging each with its reverse complement: 2,080
  features) and fit ridge regression ([Module 07](07_ridge_lasso_sparse_regression.ipynb)) — the idea behind gapped-k-mer
  SVMs, which remained competitive with early deep models.

⚠️ **Chromosome hold-out.** We train on chromosomes 1–6, choose hyper-parameters on chromosome 7 and test on chromosome 8.
Random splits of real genomes leak information: overlapping windows, repeats, duplicated genes and neighbouring
regulatory elements share sequence, so a model can score well by memorising ([Module 23](23_model_selection.ipynb)'s leakage). Our simulated
sequences are independent, so here the split mainly builds the right habit.

In [ ]:
# ── 2.4 Chromosome split, noise ceiling and the oracle-feature reference ──
train_idx = np.flatnonzero(np.isin(chrom, [f"chr{i}" for i in range(1, 7)]))
valid_idx, test_idx = np.flatnonzero(chrom == "chr7"), np.flatnonzero(chrom == "chr8")
oracle = np.array([regulatory_activity(s) for s in seqs])          # noise-free activity (answer key)


def evaluate(pred, y):
    """Pearson r and Spearman rho between predictions and measurements."""
    return float(np.corrcoef(pred, y)[0, 1]), float(stats.spearmanr(pred, y)[0])


results = {"noise ceiling (true activity)": evaluate(oracle[test_idx], activity[test_idx])}
feat = np.array([[len(find_motif_sites(s, cons)) for cons, _ in REGULATORY_MOTIFS.values()] + [g]
                 for s, g in zip(seqs, gc)])
A = np.column_stack([np.ones(N_SEQ), feat])
w_or = np.linalg.lstsq(A[train_idx], activity[train_idx], rcond=None)[0]
results["oracle motif counts + GC, linear"] = evaluate(A[test_idx] @ w_or, activity[test_idx])
print(f"train / valid / test = {len(train_idx)} / {len(valid_idx)} / {len(test_idx)} sequences")
print("fitted effects (intercept, AP-1, GATA, SP1, E-box, REST, GC):", np.round(w_or, 2))

In [ ]:
# ── 2.5 The k-mer baseline: canonical 6-mer counts + ridge regression ──
def base_index(seq_list):
    """Equal-length DNA strings → integer array (N, L) with A, C, G, T = 0, 1, 2, 3."""
    lut = np.zeros(256, np.int64)
    for i, b in enumerate(BASES):
        lut[ord(b)] = i
    return lut[np.frombuffer("".join(seq_list).encode(), np.uint8).reshape(len(seq_list), -1)]


def kmer_counts(seq_list, k=6):
    """Counts of all k-mers, each merged with its reverse complement (4^k / 2 + palindromes columns)."""
    idx = base_index(seq_list)
    n, L = idx.shape
    code = np.zeros((n, L - k + 1), np.int64)
    for u in range(k):                                         # base-4 number of each window
        code = code * 4 + idx[:, u:L - k + 1 + u]
    allk = np.arange(4 ** k)
    digits = (allk[:, None] // 4 ** np.arange(k - 1, -1, -1)) % 4
    rc = ((3 - digits[:, ::-1]) * 4 ** np.arange(k - 1, -1, -1)).sum(axis=1)
    canon, col = np.unique(np.minimum(allk, rc), return_inverse=True)
    C = np.zeros((n, len(canon)), np.float32)
    np.add.at(C, (np.repeat(np.arange(n), code.shape[1]), col[code.ravel()]), 1)
    return C


def fit_kmer_ridge(rows, lams=(300, 1000, 3000, 10000)):
    """Ridge on standardised k-mer counts of `rows`; λ chosen on chromosome 7. Returns test predictions and λ."""
    mu, sd = Ck[rows].mean(0), Ck[rows].std(0) + 1e-6            # training statistics only
    Z_tr = ((Ck[rows] - mu) / sd).astype(np.float64)
    Z_va, Z_te = (Ck[valid_idx] - mu) / sd, (Ck[test_idx] - mu) / sd
    G, b = Z_tr.T @ Z_tr, Z_tr.T @ (activity[rows] - activity[rows].mean())
    fits = [(lam, np.linalg.solve(G + lam * np.eye(len(G)), b)) for lam in lams]
    lam, w = max(fits, key=lambda t: np.corrcoef(Z_va @ t[1], activity[valid_idx])[0, 1])
    return Z_te @ w, lam


t0 = time.perf_counter()
Ck = kmer_counts(seqs)                                        # float32 counts, 6,000 × 2,080
pred_kmer, lam_k = fit_kmer_ridge(train_idx)
results["6-mer counts, ridge"] = evaluate(pred_kmer, activity[test_idx])
print(f"{Ck.shape[1]} canonical 6-mers; λ = {lam_k} chosen on chr7; {time.perf_counter() - t0:.1f} s")
pd.DataFrame(results, index=["Pearson r", "Spearman ρ"]).T.round(3)

🔍 **What just happened?** The noise ceiling on chromosome 8 is $r=0.978$: no model can do better. The oracle-feature
model reaches 0.934 — knowing the motifs gets you far — and its fitted coefficients resemble effect sizes (AP-1 0.98,
GATA 0.67, SP1 0.42, E-box 0.63, REST −1.22, GC 2.09). The k-mer ridge model, which knows nothing about motifs, reaches
0.900: 6-mers capture most motif occurrences but split each motif into overlapping pieces and know nothing about position
or combinations. The gap between 0.900 and 0.978 is the room left for a model that can learn **the motifs themselves and
their grammar**.

✏️ **Your turn.** Refit the oracle-feature model with an extra feature: the number of AP-1 sites in the central 100 bp
(`find_motif_sites(s[50:150], ...)`). Does the test correlation improve? What would that tell you about the grammar?

---
## 3 · A convolutional network for sequence → activity

> 📍 **Where we are.** Module 22's CNN had one convolution layer, global max pooling and a classification output. Here
> the target is a continuous activity driven by several motifs, their positions and their combinations, so we design
> the architecture around those biological facts.

**Architecture** (≈ 17,000 parameters):

1. **Motif scanners.** `Conv1d(4 → 64, width 9)` — 64 learnable PWMs (Module 22: *a filter is a PWM*), followed by
   ReLU, so each filter reports how strongly its motif is present at each position.
2. **Counting, with coarse position.** Average-pool every filter track into **10 bins of 20 bp**: the result says how
   much of each motif is present *and roughly where* (position effects).
3. **Motif pairs.** Max-pool the filter tracks by 5 and apply a second convolution `Conv1d(64 → 32, width 7)`, which sees
   7 × 5 + 8 ≈ 43 bp of DNA and can detect two motifs close together (combinations); average-pool into 10 bins too.
4. **Read-out.** One linear layer combines the (64 + 32) × 10 pooled features into the activity.

🧮 **Math refresher — reverse complements in one-hot code.** With channels ordered A, C, G, T, complementing a base
(A↔T, C↔G) *reverses the channel order*; reversing the strand reverses positions. So the reverse complement of a one-hot
array $x$ (4 × L) is simply `x.flip(0).flip(1)` — flip both axes. TFs bind double-stranded DNA, so an enhancer read on
the other strand should have the same activity. We use this as **data augmentation**: in each mini-batch, a random half
of the sequences is replaced by its reverse complement. (Architectures with reverse-complement-equivariant filters are an
alternative; see the 🚀 Bonus.)

Training: mean-squared error on standardised activity, AdamW with weight decay, a one-cycle learning-rate schedule
(warm up, then anneal), mini-batches of 32, and **early stopping**: keep the weights with the best validation
correlation on chromosome 7.

> 🔗 **Builds on — [Module 22](22_neural_networks.ipynb) (Sections 6–8)** for mini-batch training, weight decay and early
> stopping, and **[Module 23](23_model_selection.ipynb)** for the three-way split: chromosome 7 plays the role of the
> validation fold used for every choice (epochs, λ of the baselines), chromosome 8 is touched only for the final numbers.

In [ ]:
# ── 3.1 One-hot tensors for PyTorch: (N, 4 channels, L) ──
def one_hot_batch(seq_list):
    """Vectorised one_hot_dna for equal-length sequences → float32 array (N, 4, L)."""
    return np.eye(4, dtype=np.float32)[base_index(seq_list)].transpose(0, 2, 1).copy()


X_all = one_hot_batch(seqs)
y_mean, y_sd = activity[train_idx].mean(), activity[train_idx].std()
Y_all = ((activity - y_mean) / y_sd).astype(np.float32)        # standardised with TRAINING statistics
Xt, Yt = torch.from_numpy(X_all), torch.from_numpy(Y_all)
print("one-hot tensor:", tuple(Xt.shape), "| agrees with one_hot_dna:", np.allclose(X_all[0].T, one_hot_dna(seqs[0])))
x_rc = Xt[:1].flip(1).flip(2)
print("reverse complement via flips == reverse_complement():",
      np.allclose(x_rc[0].numpy().T, one_hot_dna(reverse_complement(seqs[0]))))

In [ ]:
# ── 3.2 The model ──
class EnhancerCNN(nn.Module):
    """conv(4→64, w=9) + ReLU = motif scanners → (a) average-pool into 10 position bins,
    (b) max-pool 5 → conv(64→32, w=7) + ReLU (motif pairs within ≈ 40 bp) → average-pool into 10 bins
    → one linear read-out of all pooled features → activity (standardised units)."""

    def __init__(self, n_filters=64, width=9, n_pair=32, n_bins=10):
        super().__init__()
        self.conv1 = nn.Conv1d(4, n_filters, width, padding=width // 2)
        self.conv2 = nn.Conv1d(n_filters, n_pair, 7, padding=3)
        self.n_bins = n_bins
        self.readout = nn.Linear((n_filters + n_pair) * n_bins, 1)

    def forward(self, x):
        h1 = F.relu(self.conv1(x))                                    # (B, 64, L): motif tracks
        h2 = F.relu(self.conv2(F.max_pool1d(h1, 5)))                  # (B, 32, L/5): motif-pair tracks
        z = torch.cat([F.adaptive_avg_pool1d(h1, self.n_bins).flatten(1),
                       F.adaptive_avg_pool1d(h2, self.n_bins).flatten(1)], dim=1)
        return self.readout(z).squeeze(1)


def predict(model, X, batch=500):
    """Predictions in ACTIVITY units for a float32 array X of shape (n, 4, L)."""
    model.eval()
    with torch.no_grad():
        out = [model(torch.from_numpy(X[s:s + batch])).numpy() for s in range(0, len(X), batch)]
    return np.concatenate(out) * y_sd + y_mean


print(f"EnhancerCNN has {sum(p.numel() for p in EnhancerCNN().parameters()):,} parameters")

In [ ]:
# ── 3.3 Training with reverse-complement augmentation and early stopping on chromosome 7 ──
def train_cnn(rows, epochs=12, lr=5e-3, batch=32, seed=SEED):
    """Train an EnhancerCNN on sequences `rows`; returns (model with best validation weights, validation r per epoch)."""
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    model = EnhancerCNN()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * int(np.ceil(len(rows) / batch)))
    best_r, best_state, history = -np.inf, None, []
    for _ in range(epochs):
        model.train()
        perm = rows[torch.randperm(len(rows), generator=gen).numpy()]
        for s in range(0, len(perm), batch):
            b = perm[s:s + batch]
            flip = torch.rand(len(b), generator=gen) < 0.5                  # reverse-complement half the batch
            xb = torch.where(flip[:, None, None], Xt[b].flip(1).flip(2), Xt[b])
            loss = F.mse_loss(model(xb), Yt[b])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        r_val = evaluate(predict(model, X_all[valid_idx]), activity[valid_idx])[0]
        history.append(r_val)
        if r_val > best_r:
            best_r, best_state = r_val, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history


t0 = time.perf_counter()
cnn, hist = train_cnn(train_idx)
print(f"trained in {time.perf_counter() - t0:.0f} s on CPU | validation r per epoch:", np.round(hist, 3))
pred_cnn = predict(cnn, X_all[test_idx])
results["CNN (this notebook)"] = evaluate(pred_cnn, activity[test_idx])
pd.DataFrame(results, index=["Pearson r", "Spearman ρ"]).T.round(3)

🔍 **What just happened?** In about 15 s on one CPU thread (in our test run) the CNN reaches a test correlation of
0.961 (Spearman 0.955) on the held-out chromosome — above the k-mer model (0.900) *and* above the oracle-feature model
that was handed the true motifs (0.934), and close to the noise ceiling (0.978). The validation curve shows the typical shape: slow for the first epochs while the filters are
random, then a rapid rise once they lock onto motifs, then a plateau as the learning rate anneals. Beating the oracle
features means the network has learned something *beyond* motif counts — position or combinations; Sections 4–6 find out
what.

🧪 **Ground truth.** The noise ceiling is computed from the true, noise-free activity — a luxury of simulation. On real
data you would compare with the correlation between replicate experiments.

Is the deep model always better? A **learning curve** answers the practical question "how much data do I need?".

In [ ]:
# ── 3.4 Learning curve: CNN vs k-mer ridge as a function of training-set size ──
t0 = time.perf_counter()
curve = {"training sequences": [], "CNN": [], "6-mer ridge": []}
for n_train, epochs in [(500, 38), (1500, 13), (len(train_idx), None)]:
    rows = train_idx[:n_train]
    if epochs is None:
        model_n = cnn                                         # the full model trained above
    else:                                                     # ~600 gradient steps for the smaller sets
        model_n, _ = train_cnn(rows, epochs=epochs)
    curve["training sequences"].append(n_train)
    curve["CNN"].append(evaluate(predict(model_n, X_all[test_idx]), activity[test_idx])[0])
    curve["6-mer ridge"].append(evaluate(fit_kmer_ridge(rows)[0], activity[test_idx])[0])
curve = pd.DataFrame(curve).set_index("training sequences")
print(f"{time.perf_counter() - t0:.0f} s")

fig, ax = plt.subplots(figsize=(5.8, 3.2))
ax.plot(curve.index, curve["CNN"], "o-", color="tab:blue", label="CNN")
ax.plot(curve.index, curve["6-mer ridge"], "s-", color="tab:gray", label="6-mer ridge")
ax.axhline(results["noise ceiling (true activity)"][0], color="tab:red", ls="--", label="noise ceiling")
ax.axhline(results["oracle motif counts + GC, linear"][0], color="tab:green", ls=":", label="oracle motif counts")
ax.set_xscale("log"); ax.set_xticks(curve.index); ax.set_xticklabels([f"{n:,}" for n in curve.index]); ax.minorticks_off()
ax.set_xlabel("training sequences (log scale)"); ax.set_ylabel("test Pearson r (chr8)")
ax.set_title("deep models need data (SIMULATED MPRA)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
curve.round(3)

🔍 **What just happened?** With 500 training sequences the k-mer ridge model (r = 0.79) beats the CNN by far (0.24 —
its filters never found the motifs); with 1,500 the CNN has nearly caught up (0.83 vs 0.86); with all 4,537 it is clearly
ahead (0.96 vs 0.90). This crossover is the norm, not the exception: a CNN must
*discover* its features, and that takes data, while a linear model on hand-made features is data-efficient but capped
by what its features can express. In real genomics the lesson is to always report a strong simple baseline — and to
remember that the advantage of deep models grew with the size of ENCODE-scale data sets.

✏️ **Your turn.** Retrain without reverse-complement augmentation (replace the `torch.where(...)` line by `xb = Xt[b]`)
on the full training set. How much does the test correlation drop? Why does augmentation help most when data are scarce?

### 📄 Research connection — Deep learning on regulatory DNA

- **What the field does.** **DeepBind** (Alipanahi *et al.* 2015) trained CNNs on protein-binding microarrays, ChIP-seq and SELEX data, one
model per factor; **DeepSEA** (Zhou & Troyanskaya 2015) predicted 919 chromatin features (TF binding, DNase
accessibility, histone marks) from 1 kb of sequence in one multi-task network and used it to score non-coding
variants; **Basset** (Kelley, Snoek & Rinn 2016) learned accessibility in 164 cell types from 600 bp; **Basenji**
(Kelley *et al.* 2018) used dilated convolutions to read 131 kb and predict thousands of genomic tracks at
128-bp resolution. DeepSTARR (de Almeida *et al.* 2022) applied the same recipe to genome-wide STARR-seq enhancer
activity and designed synthetic enhancers.
- **What this notebook builds.** A compact two-layer CNN with position-binned pooling, reverse-complement augmentation, early stopping and a
chromosome hold-out, benchmarked against a noise ceiling, oracle features and a k-mer ridge model over several
training-set sizes. Production models are larger (millions of parameters), multi-task (hundreds to thousands of
assays and cell types at once), trained on GPUs for days, and predict base-resolution profiles rather than one
number per sequence.
- **Where the frontier is.** Current directions include base-resolution profile models (BPNet, ChromBPNet) that separate enzyme bias from TF
footprints, models trained on MPRA libraries of designed or random sequences that probe grammar directly, cell-type-
and single-cell-resolved models, and careful benchmarking: on several tasks simple baselines remain competitive,
and gains on held-out chromosomes do not always translate into accurate predictions of genetic variation between
individuals (Section 8).
- **Read next.** Zhou & Troyanskaya 2015, *Predicting effects of noncoding variants with deep learning-based sequence model*;
Kelley, Snoek & Rinn 2016, *Basset: learning the regulatory code of the accessible genome with deep convolutional
neural networks*; Eraslan *et al.* 2019, *Deep learning: new computational modelling techniques for genomics*.

---
## 4 · Opening the box: what did the filters learn?

> 📍 **Where we are.** The CNN predicts well. Is it right for the right reasons? Because a first-layer filter is a
> learnable PWM, we can read it like one — but raw filter weights are hard to compare with motifs (arbitrary scale,
> negative weights, bias). The standard approach (used for Basset and DeepBind) is to collect the DNA windows that make
> each filter fire strongly and build a PWM from them.

**Activation-based PWMs.** For filter $f$, find all positions where its activation exceeds half of its maximum over a
set of sequences, take the 9-bp windows under the filter, align and count them → a PWM per filter. Then compare each
filter-PWM with the five true motifs (as IUPAC probability matrices) by the best Pearson correlation over all offsets and
both strands.

In [ ]:
# ── 4.1 Comparing a PWM with the true motifs (IUPAC → probability matrix; best correlation over offsets and strands) ──
def iupac_matrix(consensus):
    """IUPAC consensus → probability matrix (K × 4): e.g. S = C or G → 0.5 / 0.5."""
    P = np.zeros((len(consensus), 4))
    for u, b in enumerate(consensus):
        for c in IUPAC[b]:
            P[u, BASES.index(c)] = 1 / len(IUPAC[b])
    return P


def best_motif_match(P, min_overlap=5):
    """Best Pearson correlation of PWM P with any true motif, over offsets and both strands → (name, strand, r)."""
    best = ("none", "", -1.0)
    for name, (cons, _) in REGULATORY_MOTIFS.items():
        for strand, c in (("+", cons), ("−", reverse_complement(cons))):
            M = iupac_matrix(c)
            for off in range(-len(M) + min_overlap, len(P) - min_overlap + 1):
                a0, a1 = max(0, off), min(len(P), off + len(M))
                r = np.corrcoef(P[a0:a1].ravel(), M[a0 - off:a1 - off].ravel())[0, 1]
                if r > best[2]:
                    best = (name, strand, r)
    return best


print("example: the consensus TGACTCA as a PWM best matches {} on strand {} with r = {:.2f}".format(
    *best_motif_match(iupac_matrix("TGACTCA"))))

In [ ]:
# ── 4.2 Activation-based PWMs of the 64 first-layer filters and their best-matching true motif ──
sample = train_idx[:1500]
with torch.no_grad():
    H = F.relu(cnn.conv1(Xt[sample])).numpy()                      # (1500, 64 filters, 200 positions)
K_W, pad = cnn.conv1.kernel_size[0], cnn.conv1.kernel_size[0] // 2
readout = cnn.readout.weight.detach().numpy()[0][:64 * 10].reshape(64, 10)
rows = []
for f in range(H.shape[1]):
    i, p = np.nonzero(H[:, f, :] > 0.5 * H[:, f, :].max())
    keep = (p - pad >= 0) & (p - pad + K_W <= L_SEQ)
    if keep.sum() < 20:
        rows.append({"filter": f, "windows": int(keep.sum()), "match": "—", "r": np.nan, "IC (bits)": np.nan,
                     "read-out weight": readout[f].mean()})
        continue
    win = np.stack([X_all[sample[a], :, b - pad:b - pad + K_W].T for a, b in zip(i[keep], p[keep])])
    P = (win.sum(0) + 0.5) / (win.sum(0) + 0.5).sum(1, keepdims=True)
    name, strand, r = best_motif_match(P)
    rows.append({"filter": f, "windows": int(keep.sum()), "match": f"{name} ({strand})", "r": r,
                 "IC (bits)": (2 + np.sum(P * np.log2(P), axis=1)).sum(), "read-out weight": readout[f].mean(), "P": P})
filt = pd.DataFrame(rows)
del H                                                         # free the activation array (≈ 80 MB)
matched = filt[filt.r > 0.9]
print(f"filters with ≥ 20 strong activations: {filt.r.notna().sum()} of 64 | matching a true motif with r > 0.9: {len(matched)}")
print("filters per motif (r > 0.9):", matched["match"].str.split(" ").str[0].value_counts().to_dict())
filt.drop(columns="P").sort_values("r", ascending=False).head(12).round(3)

In [ ]:
# ── 4.3 🧪 Ground-truth check: the best filter for each true motif, drawn as logos ──
fig, axes = plt.subplots(2, 5, figsize=(12, 3.8))
for k, (name, (cons, effect)) in enumerate(REGULATORY_MOTIFS.items()):
    cand = filt[filt["match"].str.startswith(name + " ")].sort_values("r", ascending=False)
    draw_logo(axes[0, k], iupac_matrix(cons) * 0.98 + 0.005, f"true {name}: {cons} (effect {effect:+.1f})")
    if len(cand):
        row = cand.iloc[0]
        draw_logo(axes[1, k], row["P"], f"filter {row['filter']}: r = {row['r']:.2f}, {row['match'].split()[-1]}")
    else:
        axes[1, k].set_title(f"no filter matches {name}", fontsize=9)
plt.suptitle("top: true motifs (IUPAC) · bottom: best-matching LEARNED filter (activation-based PWM; (−) = reverse strand)",
             fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Of the 64 filters, 55 fire strongly on at least 20 windows, and 34 of those match one of the
five true motifs with $r>0.9$ (16 REST, 7 GATA, 5 AP-1, 4 SP1, 2 E-box) — several filters per motif; for the
non-palindromic motifs separate filters cover the two strands (REST, for instance, has filters for `TTCAGCACC` and for
its reverse complement `GGTGCTGAA`), since nothing forces one filter to recognise both orientations. Every one of the five motifs, including the
repressor REST, has at least one near-perfect filter (bottom row). The network rediscovered the vocabulary of the
assay from sequences and activities alone. Filters with no strong activations are "unused" — capacity the model did not
need. Note how many filters went to REST, the motif with the largest effect: the network spends capacity where the
loss is.

⚠️ **Filters are not motifs one-to-one.** Real CNNs often split a motif across several filters, learn partial motifs or
combinations, and spread one factor's signal over redundant filters. That is why attribution methods (Section 5) and
motif-discovery tools that work on attributions (TF-MoDISco) are preferred to reading filters one by one.

✏️ **Your turn.** In `train_cnn`, replace `EnhancerCNN()` by `EnhancerCNN(n_filters=8)` and retrain on the full training
set, then rerun 4.2–4.3. With only 8 motif scanners for 5 motifs × 2 strands, which motifs does the network keep, which
does it drop, and what happens to the test correlation? (Hint: look at the motif effects in `REGULATORY_MOTIFS`.)

---
## 5 · Attribution: which bases matter in a given sequence?

> 📍 **Where we are.** Filters describe the model's vocabulary globally. **Attribution** asks a local question: for this
> sequence, which positions drive the prediction, and in which direction?

🧮 **Math refresher — two attribution methods.**

- **In-silico mutagenesis (ISM).** Mutate each position to each of the three other bases and record the change in
  prediction, $\Delta_{p,b}=f(x^{(p\to b)})-f(x)$. A position's importance is the average loss of activity when its
  reference base is destroyed, $I_p=-\frac13\sum_{b\neq x_p}\Delta_{p,b}$ — positive for activator motifs, negative for
  repressor motifs. Exact, model-agnostic, but costs $3L$ forward passes per sequence.
- **Gradient × input (saliency).** A first-order Taylor expansion $f(x+\delta)\approx f(x)+\nabla_xf\cdot\delta$ predicts all
  mutation effects from one backward pass. For one-hot input, $\sum_c x_{p,c}\,\partial f/\partial x_{p,c}$ is the
  gradient at the observed base (Simonyan *et al.* 2014; DeepLIFT, Shrikumar *et al.* 2017, and SHAP replace the
  gradient by differences to a reference sequence, which behaves better for saturated units).

*In words:* ISM actually performs the mutation experiment in the computer; gradient × input estimates it from the local
slope. For a model with ReLUs and pooling they can disagree where the model is non-linear — e.g. a second motif that
matters only when the first one is present.

In [ ]:
# ── 5.1 In-silico mutagenesis and gradient × input ──
def ism(model, x):
    """Predicted effect (activity units) of every SNV of one-hot sequence x (4, L) → array (L, 4), 0 at the reference."""
    L = x.shape[1]
    ref = x.argmax(axis=0)
    alt = np.array([[b for b in range(4) if b != ref[p]] for p in range(L)])     # (L, 3) alternative bases
    pos = np.repeat(np.arange(L), 3)
    muts = np.repeat(x[None], 3 * L, axis=0)
    muts[np.arange(3 * L), :, pos] = 0
    muts[np.arange(3 * L), alt.ravel(), pos] = 1
    effects = np.zeros((L, 4))
    effects[pos, alt.ravel()] = predict(model, muts) - predict(model, x[None])[0]
    return effects


def grad_x_input(model, x):
    """Gradient of the prediction w.r.t. the one-hot input, taken at the observed base (activity units)."""
    model.eval()
    xt = torch.from_numpy(x[None].copy()).requires_grad_(True)
    model(xt).sum().backward()
    return (xt.grad[0] * xt[0].detach()).sum(0).numpy() * y_sd


def motif_track(seq):
    """For each position, the name of the true motif covering it ('' if none) — from exact matches on both strands."""
    track = np.full(len(seq), "", dtype=object)
    for name, (cons, _) in REGULATORY_MOTIFS.items():
        for s in find_motif_sites(seq, cons):
            track[s:s + len(cons)] = name
    return track


i_ex = test_idx[int(np.argmax([len(set(motif_track(seqs[i]))) for i in test_idx[:60]]))]  # a sequence with many motifs
D_ex = ism(cnn, X_all[i_ex])
imp_ex, gxi_ex, trk_ex = -D_ex.sum(1) / 3, grad_x_input(cnn, X_all[i_ex]), motif_track(seqs[i_ex])
print(f"example test sequence {i_ex}: measured activity {activity[i_ex]:.2f}, CNN {predict(cnn, X_all[i_ex:i_ex + 1])[0]:.2f}, "
      f"true {oracle[i_ex]:.2f} | motifs present: {sorted(set(trk_ex) - {''})}")

In [ ]:
# ── 5.2 Attribution maps of the example sequence, with the true motif sites shaded ──
colors = {m: plt.cm.tab10(k) for k, m in enumerate(motifs)}
fig, axes = plt.subplots(3, 1, figsize=(12, 5.2), sharex=True, gridspec_kw={"height_ratios": [1.2, 1, 1]})
v = np.abs(D_ex).max()
axes[0].imshow(D_ex.T, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v, interpolation="nearest")
axes[0].set_yticks(range(4)); axes[0].set_yticklabels(list(BASES))
axes[0].set_title("ISM: predicted change in activity when each position is mutated to each base (red = up, blue = down)")
for ax, track, label in [(axes[1], imp_ex, "ISM importance $I_p$"), (axes[2], gxi_ex, "gradient × input")]:
    ax.bar(np.arange(L_SEQ), track, width=1.0, color=np.where(track > 0, "tab:red", "tab:blue"))
    ax.set_ylabel(label, fontsize=8)
    for name in motifs:
        for p in np.flatnonzero(trk_ex == name):
            ax.axvspan(p - 0.5, p + 0.5, color=colors[name], alpha=0.15, lw=0)
for name in sorted(set(trk_ex) - {""}):
    axes[1].plot([], [], color=colors[name], lw=6, alpha=0.4, label=name)
axes[1].legend(fontsize=7, ncol=5, loc="upper right"); axes[2].set_xlabel("position (bp)")
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.3 🧪 Ground-truth check over 60 test sequences: do attributions find the true motif positions? ──
def auroc(scores, labels):
    """Area under the ROC curve via ranks (Mann–Whitney U): P(score of a positive > score of a negative)."""
    r = stats.rankdata(scores)
    n1, n0 = labels.sum(), len(labels) - labels.sum()
    return (r[labels].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


t0 = time.perf_counter()
ism_maps, imp, gxi, trk, centre = [], [], [], [], []
for i in test_idx[:60]:
    D = ism(cnn, X_all[i])
    ism_maps.append(D)
    imp.append(-D.sum(1) / 3); gxi.append(grad_x_input(cnn, X_all[i])); trk.append(motif_track(seqs[i]))
    centre.append((np.arange(L_SEQ) >= 50) & (np.arange(L_SEQ) < 150))
imp, gxi, trk, centre = map(np.concatenate, (imp, gxi, trk, centre))
in_motif = trk != ""
print(f"ISM of 60 sequences ({60 * 3 * L_SEQ:,} mutants) in {time.perf_counter() - t0:.1f} s; "
      f"{in_motif.mean():.1%} of positions lie in a true motif")
print(f"AUROC for motif positions: |ISM importance| {auroc(np.abs(imp), in_motif):.3f} | "
      f"|gradient × input| {auroc(np.abs(gxi), in_motif):.3f} | correlation ISM vs G×I: {np.corrcoef(imp, gxi)[0, 1]:.2f}")
tab = pd.DataFrame({m: {"true effect": REGULATORY_MOTIFS[m][1], "mean ISM importance": imp[trk == m].mean(),
                        "centre / flank importance": imp[(trk == m) & centre].mean() / imp[(trk == m) & ~centre].mean()}
                    for m in motifs}).T
tab.loc["(background)"] = [0.0, imp[~in_motif].mean(), np.nan]
tab.round(3)

🔍 **What just happened?** In the example sequence the ISM map lights up exactly in the shaded motif sites: mutating an
activator site lowers activity (blue in the map, positive importance), mutating the REST site *raises* it (negative
importance) — the model knows REST is a repressor. The tallest importances of all sit where a GATA site is directly
followed by two AP-1 sites (around positions 46–66): a cooperative arrangement, where destroying either site also costs the pair's bonus. Over 60
test sequences, |ISM importance| separates motif positions
from background with AUROC 0.995; gradient × input does almost as well (0.970), and the two agree closely (r = 0.86).
The mean importance per motif base follows the true effects (AP-1 0.99 > E-box 0.61 ≈ GATA 0.56 > SP1 0.26 > 0 > REST
−0.94), and bases inside every motif matter about 1.6–1.7 times more in the central half of the sequence than in the
flanks — the model has learned a
**position effect** (the true ratio is revealed in Section 6).

✏️ **Your turn.** Compute the AUROC separately for positions inside REST sites versus background, using the *signed*
score `-imp` (large where mutating the base *raises* activity, i.e. at repressor sites). What value do you expect, and
what do you get with `+imp`?

### 📄 Research connection — Interpretability and motif syntax

- **What the field does.** Attribution methods for genomic models include ISM, saliency maps (Simonyan *et al.* 2014), DeepLIFT
(Shrikumar, Greenside & Kundaje 2017), integrated gradients and DeepSHAP; TF-MoDISco clusters high-attribution
"seqlets" across thousands of sequences into consolidated motifs. **BPNet** (Avsec *et al.* 2021) trained on
base-resolution ChIP-nexus profiles of pluripotency factors and used attributions plus in-silico experiments to
reveal **soft motif syntax**: preferred spacings (including helical ~10.5-bp periodicity) and cooperative
interactions, some validated with CRISPR. Koo & Eddy (2019) showed how architecture choices (e.g. pooling) decide
whether first-layer filters learn whole motifs or parts. Novakovsky *et al.* (2023) review explainable AI in
genomics.
- **What this notebook builds.** Activation-based filter PWMs matched to the true motifs, ISM and gradient × input with an AUROC against the
ground-truth motif positions, signed importances that recover the repressor, and (next section) BPNet-style
in-silico experiments on neutral backgrounds compared with the true grammar.
- **Where the frontier is.** Making attributions robust (reference choice, gradient correction for one-hot inputs, attribution noise and
adversarial fragility), distinguishing what the model *uses* from what the biology *does*, scaling global
interpretation (TF-MoDISco-lite, in-silico marginalisation), and designing experiments — MPRAs of designed
sequences, CRISPR perturbations — to validate predicted syntax are active topics.
- **Read next.** Avsec *et al.* 2021, *Base-resolution models of transcription-factor binding reveal soft motif syntax* (Nat Genet);
Shrikumar, Greenside & Kundaje 2017, *Learning important features through propagating activation differences*;
Novakovsky *et al.* 2023, *Obtaining genetics insights from deep learning via explainable artificial intelligence*.

---
## 6 · The grammar by in-silico experiments — and opening the answer key

> 📍 **Where we are.** A trained model is a cheap simulator of the assay. Like BPNet's authors, we now design
> sequences that never occurred in the training data — two motifs at controlled spacing, a motif sliding along the
> sequence, a motif flipped to the other strand — and ask the model what it predicts. Then, finally, we open the answer
> key and compare.

**The answer key.** Here is the docstring of `regulatory_activity`, the rule that generated every measurement:

In [ ]:
# ── 6.1 Open the answer key ──
print(regulatory_activity.__doc__)

So the true grammar has three parts beyond "each motif adds its effect": a **position effect** (sites in the central
half count fully, flanking sites 60 %), **cooperativity** (+1.0 if an AP-1 and a GATA site start within 30 bp of each
other) and a **GC effect**. Orientation does not matter. The oracle-feature model of Section 2 knew the motifs but none of
these rules. What did the CNN learn?

The experiments use **neutral backgrounds**: random 200-bp sequences with equal base frequencies that contain no match to
any of the five motifs. We insert the consensus `TGACTCA` (AP-1) and `AGATAA` (GATA) and average predictions over 60
backgrounds; the oracle tells us the truth for the same sequences.

In [ ]:
# ── 6.2 Neutral backgrounds and motif insertion ──
rng = np.random.default_rng(SEED + 6)


def neutral_backgrounds(n, length=L_SEQ):
    """Random sequences (equal base frequencies) with no exact match to any of the five motifs, on either strand."""
    out = []
    while len(out) < n:
        s = "".join(rng.choice(list(BASES), length))
        if all(len(find_motif_sites(s, cons)) == 0 for cons, _ in REGULATORY_MOTIFS.values()):
            out.append(s)
    return out


def insert(seq, motif, pos):
    """Overwrite seq[pos : pos + len(motif)] with motif."""
    return seq[:pos] + motif + seq[pos + len(motif):]


def compare(seq_list):
    """Mean CNN prediction and mean true (oracle) activity over a list of designed sequences."""
    return predict(cnn, one_hot_batch(seq_list)).mean(), np.mean([regulatory_activity(s) for s in seq_list])


backgrounds = neutral_backgrounds(60)
AP1, GATA = "TGACTCA", "AGATAA"
base_pred, base_true = compare(backgrounds)
print(f"60 neutral backgrounds: CNN {base_pred:.2f}, true {base_true:.2f}")
for name, cons in [("AP-1", AP1), ("GATA", GATA)]:
    p, t = compare([insert(s, cons, 95) for s in backgrounds])
    print(f"+ one {name} in the centre: CNN Δ = {p - base_pred:+.2f}, true Δ = {t - base_true:+.2f}")

In [ ]:
# ── 6.3 Spacing (cooperativity), orientation and position experiments ──
spacing = np.arange(8, 82, 2)                                   # distance between the two motif starts (bp)
sp_pred, sp_true = np.array([compare([insert(insert(s, AP1, 70), GATA, 70 + d) for s in backgrounds])
                             for d in spacing]).T
fwd = compare([insert(insert(s, AP1, 70), GATA, 90) for s in backgrounds])
rev = compare([insert(insert(s, AP1, 70), reverse_complement(GATA), 90) for s in backgrounds])
positions = np.arange(0, L_SEQ - len(AP1) + 1, 6)
pos_pred, pos_true = np.array([compare([insert(s, AP1, q) for s in backgrounds]) for q in positions]).T

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.3))
axes[0].plot(spacing, sp_true, "k--", label="truth (answer key)")
axes[0].plot(spacing, sp_pred, "o-", ms=3, color="tab:blue", label="CNN prediction")
axes[0].axvline(30, color="0.6", lw=0.8)
axes[0].set_xlabel("spacing between AP-1 and GATA starts (bp)"); axes[0].set_ylabel("activity (mean of 60 backgrounds)")
axes[0].set_title("cooperativity: AP-1 + GATA at increasing distance"); axes[0].legend(fontsize=7)
axes[1].plot(positions, pos_true - base_true, "k--", label="truth")
axes[1].plot(positions, pos_pred - base_pred, "o-", ms=3, color="tab:blue", label="CNN")
axes[1].axvspan(L_SEQ / 4 - 3.5, 3 * L_SEQ / 4 - 3.5, color="0.9", zorder=0)
axes[1].set_xlabel("start of a single AP-1 site (bp)"); axes[1].set_ylabel("Δ activity vs background")
axes[1].set_title("position effect: one AP-1 site slid along the sequence"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"orientation (GATA 20 bp after AP-1): forward CNN {fwd[0]:.2f} / true {fwd[1]:.2f}; "
      f"reversed CNN {rev[0]:.2f} / true {rev[1]:.2f}")

In [ ]:
# ── 6.4 🧪 Ground-truth check: how much of each grammar rule did the CNN learn? ──
close, far = (spacing >= 10) & (spacing <= 26), (spacing >= 36) & (spacing <= 70)
centre_pos = (positions + len(AP1) / 2 >= L_SEQ / 4) & (positions + len(AP1) / 2 <= 3 * L_SEQ / 4)
grammar = pd.DataFrame({
    "cooperativity bonus (close − far pairs)": [sp_pred[close].mean() - sp_pred[far].mean(),
                                                sp_true[close].mean() - sp_true[far].mean()],
    "AP-1 effect, centre": [(pos_pred - base_pred)[centre_pos].mean(), (pos_true - base_true)[centre_pos].mean()],
    "AP-1 effect, flanks": [(pos_pred - base_pred)[~centre_pos].mean(), (pos_true - base_true)[~centre_pos].mean()],
    "orientation (reversed − forward GATA)": [rev[0] - fwd[0], rev[1] - fwd[1]]}, index=["CNN", "truth"]).T
coop_frac = np.mean([any(abs(a - g) <= 30 for a in find_motif_sites(seqs[i], REGULATORY_MOTIFS["AP-1"][0])
                         for g in find_motif_sites(seqs[i], REGULATORY_MOTIFS["GATA"][0])) for i in train_idx])
print(f"training sequences that contain a cooperative AP-1–GATA pair (starts ≤ 30 bp apart): {coop_frac:.1%}")
grammar.round(2)

🔍 **What just happened?** Three rules, three different outcomes:

- **Position effect — learned.** A single AP-1 site adds about 1.06 in the centre and 0.67 in the flanks according to the CNN,
  versus 1.0 and 0.6 in truth; the predicted curve rises and falls at roughly the right places, only smoothed, because
  the model sees position through 20-bp bins.
- **Orientation — correctly ignored.** Flipping GATA changes the prediction by only 0.01 (truth 0).
- **Cooperativity — partly learned.** The truth is a step: +1.0 when the two sites start ≤ 30 bp apart, nothing beyond.
  The CNN predicts that close pairs are more active than distant ones by 0.31 — the right sign, but less than a third of
  the true bonus, and as a gradual decline rather than a sharp step (the dip of the true curve at 78–80 bp is the GATA site
  leaving the central half). Only 10.4 % of training sequences contain a cooperative AP-1–GATA pair,
  and very few sit near the 30-bp boundary, so the data constrain the shape of the rule weakly; the network's smooth
  pooled features fill the gap with a plausible, but wrong, interpolation.

This is a general lesson for interpreting sequence models: they learn the parts of the grammar that the training data
constrain, and they extrapolate smoothly elsewhere. A designed library — many sequences with controlled spacings, as in
BPNet's validation experiments or synthetic MPRAs — would pin the rule down.

✏️ **Your turn.** Regenerate the data with `motif_rate=1.5` (half as many motifs, as in the toolkit's default), retrain,
and repeat 6.3–6.4. Does the CNN still learn the cooperativity? How does the fraction of cooperative training sequences
change?

---
## 7 · Variant-effect prediction: every possible single-letter change

> 📍 **Where we are.** Module 35's fine-mapping leaves credible sets of non-coding variants. A sequence model can score
> each of them: $\Delta = f(\text{alt}) - f(\text{ref})$. ISM already computed this for every possible single-nucleotide
> variant (SNV) of our 60 test sequences — 60 × 200 × 3 = 36,000 variants. Here we compare the predictions with the
> true effects from the answer key.

In [ ]:
# ── 7.1 Predicted vs true effects of all 36,000 SNVs of 60 test sequences ──
t0 = time.perf_counter()
pred_d, true_d, in_site, site_name = [], [], [], []
for i, D in zip(test_idx[:60], ism_maps):
    s, ref = seqs[i], X_all[i].argmax(0)
    a0 = regulatory_activity(s)
    track = motif_track(s)
    for p in range(L_SEQ):
        for b in range(4):
            if b != ref[p]:
                pred_d.append(D[p, b])
                true_d.append(regulatory_activity(s[:p] + BASES[b] + s[p + 1:]) - a0)
                in_site.append(track[p] != ""); site_name.append(track[p])
pred_d, true_d, in_site, site_name = map(np.array, (pred_d, true_d, in_site, site_name))
large = np.abs(true_d) > 0.3
top = np.argsort(-np.abs(pred_d))
print(f"{len(true_d):,} SNVs scored in {time.perf_counter() - t0:.1f} s")
print(f"near-neutral (|true Δ| ≤ 0.02, i.e. only the GC term changes): {np.mean(np.abs(true_d) <= 0.02):.1%} | "
      f"large (|Δ| > 0.3): {large.mean():.1%}, of which {np.mean(in_site[large]):.1%} inside existing motifs")
print(f"Spearman ρ all SNVs {stats.spearmanr(pred_d, true_d)[0]:.3f} | Pearson r {np.corrcoef(pred_d, true_d)[0, 1]:.3f} | "
      f"Spearman among large-effect SNVs {stats.spearmanr(pred_d[large], true_d[large])[0]:.3f}")
print(f"AUROC of |predicted Δ| for large-effect SNVs {auroc(np.abs(pred_d), large):.3f} | "
      f"precision of the top 100 / 500 predictions: {large[top[:100]].mean():.2f} / {large[top[:500]].mean():.2f}")
print(f"sign agreement among large-effect SNVs: {np.mean(np.sign(pred_d[large]) == np.sign(true_d[large])):.1%}")

In [ ]:
# ── 7.2 The variant-effect scatter plot ──
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.3, 1]})
axes[0].scatter(true_d[~in_site], pred_d[~in_site], s=2, color="0.6", alpha=0.4, label="outside true motifs")
axes[0].scatter(true_d[in_site], pred_d[in_site], s=3, color="tab:red", alpha=0.5, label="inside true motifs")
lim = [min(true_d.min(), pred_d.min()) - 0.1, max(true_d.max(), pred_d.max()) + 0.1]
axes[0].plot(lim, lim, "k--", lw=0.8)
axes[0].set_xlabel("true Δ activity (answer key)"); axes[0].set_ylabel("CNN-predicted Δ")
axes[0].set_title("all SNVs of 60 test sequences (SIMULATED)"); axes[0].legend(fontsize=7, markerscale=4)
axes[1].hist(true_d, bins=80, color="tab:blue", log=True)
axes[1].set_xlabel("true Δ activity"); axes[1].set_ylabel("number of SNVs (log)")
axes[1].set_title("most variants are (nearly) neutral")
plt.tight_layout(); plt.show()

🔍 **What just happened?** 86.8 % of the 36,000 variants are near-neutral — they only nudge the GC term by ±0.01 — and
13.0 % have a large effect (|Δ| > 0.3; many, because our library is motif-rich). Of these, 91 % lie inside existing
motifs — destroying an activator site lowers activity, destroying a REST site raises it — and the remaining 9 % *create*
a new site. The CNN ranks the variants that matter extremely
well: AUROC 0.997 for picking out large-effect variants, the top 100 (and top 500) predictions are all large-effect
variants, the sign is right for 99.9 % of them, and among large-effect variants the Spearman correlation with the truth
is 0.91. The overall Spearman correlation is much lower (0.57) — not because the model is bad but because ranking tens of thousands of
near-zero effects against each other is mostly ranking noise; the Pearson correlation (0.96), dominated by the large
effects, tells the other half of the story. **Which metric you report changes the conclusion** — always look at the
scatter plot.

In real genetics this is how sequence models are used: to prioritise the variants of a credible set for experimental
follow-up (MPRA, CRISPR editing), to interpret eQTLs, and as functional priors in fine-mapping (Module 35). The key
caveat comes next.

✏️ **Your turn.** (a) How many large-effect SNVs lie *outside* existing motifs (`large & ~in_site`)? Look at a few of
them with `motif_track` on the mutated sequence: what did these mutations create? (b) Find the large-effect SNV with
the largest prediction error `abs(pred_d - true_d)` and explain from its context why the model got it wrong.

---
## 8 · Large models and genomic language models

> 📍 **Where we are.** Our CNN reads 200 bp. Genes are regulated by enhancers that can lie tens to hundreds of
> kilobases away — the ZRS enhancer of the limb gene *SHH* sits about a megabase from its target — and the measurements
> available are not one activity per sequence but thousands of genome-wide tracks.

**Long-context supervised models.** The field has steadily increased the window of DNA a model reads and the number of
assays it predicts at once:

- **Basenji** (Kelley *et al.* 2018): dilated convolutions, 131 kb of input, thousands of ENCODE/FANTOM tracks.
- **Enformer** (Avsec *et al.* 2021, Nat Methods): convolutions + transformer layers (attention, 🔗 [Module 24](24_attention_transformers.ipynb)), 196 kb
  of input; attention lets distant positions interact directly.
- **Borzoi** (Linder *et al.* 2025): 524 kb of input, predicts RNA-seq coverage, linking sequence to splicing and
  transcript levels.
- **AlphaGenome** (Avsec *et al.*, first released as a preprint in 2025): about 1 Mb of input at base-pair resolution
  across many modalities.

**Self-supervised genomic language models** learn from unlabelled genomes by predicting masked or next nucleotides
(🔗 [Module 24](24_attention_transformers.ipynb)): **DNABERT** (Ji *et al.* 2021; k-mer tokens, BERT), the **Nucleotide Transformer** (Dalla-Torre *et al.*
2025; trained on thousands of human genomes and many species), and **Evo** (Nguyen *et al.* 2024; a 7-billion-parameter
long-context model trained on prokaryotic and phage genomes, used for zero-shot fitness prediction and design; its
successor Evo 2 was first released as a preprint in 2025). Their zero-shot variant score is the change in sequence likelihood,
$\log P(\text{alt})-\log P(\text{ref})$: variants that make the sequence "less genome-like" are flagged as damaging.

**Documented limitations.** Karollus, Mauermeier & Gagneur (2023) found that Enformer-style models capture
promoter-proximal determinants of expression well but make little use of distal enhancers. Sasse *et al.* (2023) showed
that state-of-the-art models explain little of the expression differences *between individuals* caused by their
personal genome variants — often getting even the direction wrong — although they do well across genes. Language-model
likelihoods measure evolutionary or genomic "typicality", which is related to, but not the same as, function in a given
cell type.

In [ ]:
# ── 8.1 How much DNA do sequence models read? (input lengths from the cited papers; log scale) ──
context = {"this notebook's CNN": 200, "Basset (2016)": 600, "DeepSEA (2015)": 1_000, "BPNet (2021)": 1_000,
           "Basenji (2018)": 131_072, "Enformer (2021)": 196_608, "Borzoi (2025)": 524_288,
           "AlphaGenome (2025)": 1_000_000}
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(list(context), list(context.values()), color=["tab:red"] + ["tab:blue"] * 7)
ax.axvspan(1e4, 1e6, color="tab:orange", alpha=0.12, zorder=0, label="many enhancer–promoter distances (≈ 10 kb – 1 Mb)")
ax.set_xscale("log"); ax.set_xlabel("input sequence length (bp, log scale)"); ax.invert_yaxis()
ax.set_title("sequence-to-function models: input context"); ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()

> 🔗 **Builds on — [Modules 19](19_hmm_inference.ipynb)–[20](20_hmm_learning.ipynb) (Markov chains and their
> maximum-likelihood estimation by counting).** A $k$-th-order Markov chain over A, C, G, T is the "visible" part of an
> HMM without hidden states; transformers (Module 24) replace the fixed-length context by learned attention over the
> whole sequence.

**A DNA language model you can train in a second.** The simplest language model is a **Markov chain**: the probability of each base given the previous $k$ bases, estimated by counting. We train an order-5 chain on all
training-chromosome sequences — without using any activity measurements — and score each of the 36,000 SNVs by the change
in log-likelihood. Motifs are over-represented in our "genome" (they were planted), just as functional elements are
conserved in real genomes, so destroying one should look "unusual" to the language model.

In [ ]:
# ── 8.2 Zero-shot variant scores from an order-5 Markov-chain DNA language model ──
ORDER = 5
idx_all = base_index(seqs)


def context_codes(idx):
    """Base-4 code of the ORDER preceding bases for every position ORDER..L-1 of each row of idx."""
    L = idx.shape[1]
    code = np.zeros((idx.shape[0], L - ORDER), np.int64)
    for u in range(ORDER):
        code = code * 4 + idx[:, u:L - ORDER + u]
    return code


counts = np.ones((4 ** ORDER, 4))                                   # add-one smoothing
np.add.at(counts, (context_codes(idx_all[train_idx]).ravel(), idx_all[train_idx][:, ORDER:].ravel()), 1)
log_p = np.log(counts / counts.sum(axis=1, keepdims=True))


def log_lik(idx):
    """Log-likelihood of each row of idx under the Markov chain (first ORDER bases ignored)."""
    return log_p[context_codes(idx), idx[:, ORDER:]].sum(axis=1)


lm_d = []
for i in test_idx[:60]:
    ref = idx_all[i]
    muts = np.repeat(ref[None], 3 * L_SEQ, axis=0)
    pos = np.repeat(np.arange(L_SEQ), 3)
    muts[np.arange(3 * L_SEQ), pos] = np.array([[b for b in range(4) if b != ref[p]] for p in range(L_SEQ)]).ravel()
    lm_d.append(log_lik(muts) - log_lik(ref[None])[0])              # same SNV order as in Section 7
lm_d = np.concatenate(lm_d)
print(f"zero-shot LM: AUROC for large-effect SNVs (score = −Δ log-lik) {auroc(-lm_d, large):.3f} "
      f"(CNN {auroc(np.abs(pred_d), large):.3f})")
print(f"Spearman with the SIGNED true effect: LM {stats.spearmanr(lm_d, true_d)[0]:.3f}, CNN {stats.spearmanr(pred_d, true_d)[0]:.3f}")
rest_break = large & (true_d > 0)                                   # destroying a REST site RAISES activity
print(f"large-effect SNVs that raise activity: {rest_break.sum()} ({np.mean(site_name[rest_break] == 'REST'):.0%} of them "
      f"inside REST sites) — LM calls {np.mean(lm_d[rest_break] < 0):.0%} of them 'damaging' (lower likelihood), "
      f"CNN predicts an increase for {np.mean(pred_d[rest_break] > 0):.0%}")

🔍 **What just happened?** Trained on sequence alone, the Markov-chain language model flags large-effect variants with
AUROC 0.90 — far better than chance, because breaking an over-represented word lowers the likelihood — but below the
supervised CNN (0.997). It cannot tell the **direction** of an effect: its Spearman correlation with the signed true effect
is only 0.18 (CNN 0.57), and 69 % of the 1,274 large-effect variants that *raise* activity (68 % of which break a REST
site) are scored as damaging
— the LM sees "a motif was destroyed", not "a repressor was lost". Real genomic language models are vastly more
powerful than an order-5 Markov chain, but the logic of zero-shot scoring is the same: likelihood measures how typical a
sequence is, and turning typicality into a signed, cell-type-specific functional prediction requires functional data.

✏️ **Your turn.** Change `ORDER` to 2 and to 7 and rerun 8.2. Which order flags large-effect variants best? Why does a
too-high order hurt with only 4,500 training sequences? (Think of Module 23: the number of parameters is $3\cdot4^k$.)

### 📄 Research connection — Long-range models, variant effects and their limits

- **What the field does.** Enformer (Avsec *et al.* 2021) brought attention to sequence-to-expression modelling with 196-kb inputs; Borzoi and
AlphaGenome (first released as a preprint in 2025) extended context and output modalities. Genomic language models — DNABERT, the
Nucleotide Transformer (Dalla-Torre *et al.* 2025), HyenaDNA, Evo (Nguyen *et al.* 2024) and others — are trained
without labels and used for zero-shot variant scoring, fine-tuning and sequence design. Variant-effect predictions
from these models are increasingly used as priors for fine-mapping and for interpreting rare non-coding variants.
- **What this notebook builds.** Variant-effect prediction for all SNVs of held-out sequences with an honest evaluation against true effects
(rank vs magnitude vs sign metrics), and a Markov-chain language model as the simplest zero-shot scorer, which
exposes the core limitation of likelihood-based scores (no sign, no cell type). Our 200-bp model has no
long-range interactions at all — the main thing large models add.
- **Where the frontier is.** Benchmarks show that models predicting well across genes can still fail across individuals (Sasse *et al.* 2023)
and underuse distal enhancers (Karollus *et al.* 2023); whether larger language models beat well-designed
supervised baselines on regulatory tasks is debated, and evaluation suites (and leakage between pre-training and
test data) are an active topic. Integrating 3-D genome structure, cell-type resolution (single-cell ATAC/RNA),
and allele-specific measurements, and validating predictions with large MPRA and CRISPR screens, are current
directions.
- **Read next.** Karollus, Mauermeier & Gagneur 2023, *Current sequence-based models capture gene expression determinants in
promoters but mostly ignore distal enhancers*; Sasse *et al.* 2023, *Benchmarking of deep neural networks for
predicting personal gene expression from DNA sequence highlights shortcomings*; Nguyen *et al.* 2024, *Sequence
modeling and design from molecular to genome scale with Evo*.

---
## 9 · 🧭 From toy to real data

> 📍 **Where we are.** Everything ran on 6,000 simulated sequences with a known grammar. Here is what changes on real
> data. (Reading only — nothing is downloaded.)

**Data and resources.**
- [ENCODE](https://www.encodeproject.org/) (ENCODE Project Consortium 2012 and later phases): ChIP-seq, ATAC/DNase-seq,
  histone marks and RNA-seq across hundreds of cell types; Roadmap Epigenomics; [GTEx](https://gtexportal.org/) eQTLs.
- Motif databases: [JASPAR](https://jaspar.elixir.no/) (curated, open; Castro-Mondragon *et al.* 2022) and CIS-BP
  (Weirauch *et al.* 2014).
- Reporter-assay data: MPRA and STARR-seq libraries (e.g. the genome-wide fly STARR-seq data behind DeepSTARR);
  synthetic random-sequence libraries in yeast and human cells.
- Pre-trained models and model zoos (Kipoi; the authors' releases of Basenji/Enformer/Borzoi/BPNet).

**Production tools.**

| Task | This notebook | Production tools |
|---|---|---|
| Motif scanning / discovery | PWM scan, filter PWMs | FIMO / MEME suite, HOMER, JASPAR scans; TF-MoDISco on attributions |
| Sequence models | 2-layer CNN, 200 bp | BPNet / ChromBPNet, Basenji, Enformer, Borzoi (PyTorch/TensorFlow; GPUs) |
| Attribution | ISM, gradient × input | DeepLIFT / DeepSHAP, integrated gradients (Captum), fast ISM |
| Variant effects | Δ from ISM | model-specific scoring scripts; VEP plug-ins; precomputed scores for common variants |
| Language models | order-5 Markov chain | DNABERT-2, Nucleotide Transformer, Evo / Evo 2, Caduceus (Hugging Face) |

**Compute.** Training Enformer-class models takes days on many accelerators; inference on one variant needs a 196-kb or
longer window and a GPU for reasonable speed. Base-resolution ChIP/ATAC models (BPNet) train on a single GPU in hours.

**The first three things that go wrong.**
1. **Shortcuts and biases.** Models learn GC content, Tn5 or mappability biases, repeat elements or peak-calling
   artefacts that correlate with the label. Our simulator contains a real GC effect; real data contain several spurious
   ones. Bias-correction models (as in ChromBPNet) and attribution checks are standard.
2. **Leakage.** Random splits leak through overlapping windows, paralogous genes and repeats — hold out whole chromosomes
   (and check homology between training and test regions).
3. **Over-interpretation.** Attributions are noisy and model-dependent, a filter that "looks like" a motif may be a
   correlated artefact, and in-silico experiments on unnatural sequences extrapolate. Validate with experiments (MPRA,
   CRISPR) — and remember Section 8's limits for personal-genome variants and distal enhancers.

**Design.** The same models can be run backwards to design sequences with desired activity — DeepSTARR designed
synthetic enhancers with tailored activity in fly cells (de Almeida *et al.* 2022). Generative design is the subject of
[Module 39](39_generative_design.ipynb).

## 📌 Key takeaways

- **Regulatory DNA is a grammar of motifs.** TFs bind short, fuzzy words summarised by PWMs (log-odds scores,
  information content in bits); activity depends on which motifs are present, where, and in which combinations.
- **Benchmark honestly before going deep:** chromosome hold-out, a noise ceiling, an oracle or knowledge-based reference,
  and a strong k-mer baseline. Deep models overtake simple ones only with enough data.
- **A CNN is a bank of learnable PWMs plus a read-out.** Architecture encodes biology: position-binned pooling for
  position effects, a second layer for motif pairs, reverse-complement augmentation for double-stranded DNA.
- **Opening the box works when checked against truth:** activation-based filter PWMs recovered all five motifs; ISM and
  gradient × input located motif positions (AUROC 0.97–0.995) and recovered the repressor's sign.
- **In-silico experiments probe grammar** — and show what the data did and did not constrain (position learned,
  orientation correctly ignored, cooperativity only blurred).
- **Variant effects:** $\Delta=f(\text{alt})-f(\text{ref})$ ranks large-effect variants well; overall rank correlations
  are dominated by near-neutral variants — report several metrics and look at the scatter plot.
- **Large models and language models** extend context and use unlabelled genomes, but have documented blind spots
  (distal enhancers, personal-genome variation; likelihood ≠ signed function).

---
## ✅ Checkpoint

1. A PWM column has probabilities (A, C, G, T) = (0.5, 0, 0.5, 0) (ignore pseudocounts). What is its information content? What is the IC of the whole AP-1 consensus `TGASTCA`?
   <details><summary>Answer</summary>Entropy $=1$ bit, so IC $=2-1=1$ bit. `TGASTCA` has six fixed positions (2 bits each) and one `S` (1 bit): 13 bits, so an exact match occurs by chance about once every $2^{13}\approx8{,}000$ windows per strand.</details>
2. Why do we hold out whole chromosomes instead of random sequences when evaluating models on real genomes?
   <details><summary>Answer</summary>Nearby and paralogous regions share sequence (overlapping windows, repeats, duplicated genes), so random splits put near-copies of test sequences in the training set and reward memorisation (leakage, Module 23).</details>
3. Why is a first-layer convolution filter a PWM, and what does max pooling (or average pooling) add?
   <details><summary>Answer</summary>Its output at each position is a weighted sum of the one-hot bases in a window — exactly a log-odds PWM score if the weights are log-odds (Module 22). Max pooling asks whether the motif occurs anywhere in a window (translation invariance within the window); average pooling counts how much of it there is; pooling into bins keeps coarse position.</details>
4. How do you compute the reverse complement of a one-hot sequence with channels A, C, G, T, and why is reverse-complement augmentation justified?
   <details><summary>Answer</summary>Flip both axes (`x.flip(0).flip(1)`): reversing the channel order complements the bases, reversing positions reads the other strand. TFs bind double-stranded DNA, so the same element read on either strand has the same activity.</details>
5. What does in-silico mutagenesis measure, how many forward passes does it need for a 200-bp sequence, and how does gradient × input differ?
   <details><summary>Answer</summary>The predicted change in output for each possible single-base substitution; $3\times200=600$ forward passes (plus the reference). Gradient × input estimates the same changes from one backward pass by a first-order Taylor expansion, which can be inaccurate where the model is non-linear.</details>
6. In Section 7, the overall Spearman correlation between predicted and true variant effects was much lower than the AUROC for large-effect variants. Explain.
   <details><summary>Answer</summary>Most variants have true effects near zero (only the GC term changes); ranking them against each other is ranking noise, which drags the rank correlation down, while the few large effects — which matter biologically — are ranked almost perfectly.</details>
7. Code reading: `torch.where(flip[:, None, None], Xt[b].flip(1).flip(2), Xt[b])` — what are the shapes, and what does the line do?
   <details><summary>Answer</summary>`Xt[b]` is (batch, 4, L); `flip` is a boolean vector of length batch, broadcast to (batch, 1, 1). For each sequence with `flip=True` the reverse complement (channels and positions flipped) replaces the original.</details>
8. Why can a genomic language model flag a variant as damaging even though it increases enhancer activity?
   <details><summary>Answer</summary>Its score is the change in sequence likelihood: destroying an over-represented word (here a REST repressor site) makes the sequence less typical, whatever the direction of the functional effect. Direction and cell type require functional (supervised) information.</details>

## 🧠 Questions & discussion

1. Our CNN beat the oracle-feature model that was given the true motifs. Does that mean a CNN would discover the regulatory code of a real cell type from ENCODE data? List what is different about real data.
2. Section 6 showed that the CNN learned a blurred version of the cooperativity rule. Design an MPRA library (how many sequences, which variations) that would let a model learn it exactly.
3. A sequence model predicts that a fine-mapped non-coding variant in a credible set of 12 SNPs (Module 35) disrupts a GATA site in erythroid cells. What experiments would you do before claiming it is the causal variant?
4. When would you prefer a k-mer or PWM-based model over a deep network in a real study, even if the network is a little more accurate?
5. Genomic language models are trained on genomes of many species. What could they learn that supervised models cannot, and what could make their zero-shot scores misleading for human regulatory variants?
6. Variant-effect predictions are starting to be used to interpret patients' genomes. What evidence of accuracy and calibration should be required before such predictions influence clinical decisions?

## 🚀 Bonus — going further

- **Dilated convolutions.** Replace the second layer by three dilated convolutions (dilation 2, 4, 8) and compute the receptive field. Does the cooperativity curve of 6.3 become sharper? (This is Basenji's trick for long contexts.)
- **A tiny transformer vs the CNN** (🔗 [Module 24](24_attention_transformers.ipynb)): after the first convolution, add one self-attention layer over the 40 max-pooled positions before pooling into bins. Compare test r and the cooperativity curve.
- **Reverse-complement-equivariant convolution.** Tie each filter to its reverse complement (`w_rc = w.flip(0).flip(1)`), scan with both and take the maximum; then train without augmentation. Compare with Section 3's augmented model.
- **Motif discovery from attributions.** Collect all ISM windows (9 bp) around positions with |importance| > 0.2 in the 60 test sequences, cluster them (k-means on one-hot windows, [Module 12](12_clustering_kmeans_hierarchical.ipynb), aligning both strands) and draw a logo per cluster — a mini TF-MoDISco.
- Try a real model: the public Enformer or BPNet releases come with example notebooks for variant scoring (outside this course; requires downloads and preferably a GPU).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 36 · Regulatory genomics: deep learning from DNA sequence to function
Biological question: which motifs, in which arrangements, make an enhancer active — and what does a single-letter
                     change do?
Data (simulated): simulate_regulatory_sequences (6,000 × 200 bp, motif_rate 3, 8 pretend chromosomes);
                  ground truth = regulatory_activity (motif effects, position effect, AP-1–GATA cooperativity, GC)
Method: PWM + scan; oracle-feature regression; 6-mer ridge; 2-layer CNN with binned pooling and RC augmentation;
        filter PWMs; ISM and gradient × input; in-silico grammar experiments; SNV effect prediction; Markov-chain LM
Key equation: S(s) = Σ_u log2(P_{u,s_u} / 0.25);  IC_u = 2 + Σ_c P_uc log2 P_uc;  Δ = f(alt) − f(ref)
Key parameters: filter width/number, pooling bins, epochs, learning rate, RC augmentation, Markov order
Ground-truth check result: test r CNN … vs k-mer … vs oracle features … vs ceiling …; motifs recovered …/5;
                           AUROC(ISM) …; cooperativity learned … of 1.0; variant AUROC … / Spearman …
What changed when I modified … (training size / motif_rate / augmentation / ORDER): …
One failure mode: (e.g. blurred cooperativity rule; overall Spearman dominated by neutral variants; LM ignores sign)
Connection to research (paper/tool): DeepSEA / Basset / BPNet / Enformer / TF-MoDISco / Evo
```

## 📚 References

- Alipanahi B., Delong A., Weirauch M. T., Frey B. J. (2015) Predicting the sequence specificities of DNA- and RNA-binding proteins by deep learning. *Nat Biotechnol* 33:831–838.
- Zhou J., Troyanskaya O. G. (2015) Predicting effects of noncoding variants with deep learning-based sequence model. *Nat Methods* 12:931–934.
- Kelley D. R., Snoek J., Rinn J. L. (2016) Basset: learning the regulatory code of the accessible genome with deep convolutional neural networks. *Genome Res* 26:990–999.
- Kelley D. R. *et al.* (2018) Sequential regulatory activity prediction across chromosomes with convolutional neural networks. *Genome Res* 28:739–750.
- Avsec Ž. *et al.* (2021) Base-resolution models of transcription-factor binding reveal soft motif syntax. *Nat Genet* 53:354–366.
- Avsec Ž. *et al.* (2021) Effective gene expression prediction from sequence by integrating long-range interactions. *Nat Methods* 18:1196–1203.
- de Almeida B. P., Reiter F., Pagani M., Stark A. (2022) DeepSTARR predicts enhancer activity from DNA sequence and enables the de novo design of synthetic enhancers. *Nat Genet* 54:613–624.
- Shrikumar A., Greenside P., Kundaje A. (2017) Learning important features through propagating activation differences. *Proceedings of the 34th International Conference on Machine Learning (ICML)*.
- Simonyan K., Vedaldi A., Zisserman A. (2014) Deep inside convolutional networks: visualising image classification models and saliency maps. *ICLR workshop*.
- Koo P. K., Eddy S. R. (2019) Representation learning of genomic sequence motifs with convolutional neural networks. *PLoS Comput Biol* 15:e1007560.
- Eraslan G., Avsec Ž., Gagneur J., Theis F. J. (2019) Deep learning: new computational modelling techniques for genomics. *Nat Rev Genet* 20:389–403.
- Novakovsky G., Dexter N., Libbrecht M. W., Wasserman W. W., Mostafavi S. (2023) Obtaining genetics insights from deep learning via explainable artificial intelligence. *Nat Rev Genet* 24:125–137.
- Karollus A., Mauermeier T., Gagneur J. (2023) Current sequence-based models capture gene expression determinants in promoters but mostly ignore distal enhancers. *Genome Biol* 24:56.
- Sasse A. *et al.* (2023) Benchmarking of deep neural networks for predicting personal gene expression from DNA sequence highlights shortcomings. *Nat Genet* 55:2060–2064.
- Nguyen E. *et al.* (2024) Sequence modeling and design from molecular to genome scale with Evo. *Science* 386:eado9336.
- Dalla-Torre H. *et al.* (2025) Nucleotide Transformer: building and evaluating robust foundation models for human genomics. *Nat Methods* 22:287–297.
- Ji Y., Zhou Z., Liu H., Davuluri R. V. (2021) DNABERT: pre-trained Bidirectional Encoder Representations from Transformers model for DNA-language in genome. *Bioinformatics* 37:2112–2120.
- Linder J., Srivastava D., Yuan H., Agarwal V., Kelley D. R. (2025) Predicting RNA-seq coverage from DNA sequence as a unifying model of gene regulation. *Nat Genet* 57:949–961.
- Avsec Ž. *et al.* (2025) AlphaGenome: advancing regulatory variant effect prediction with a unified DNA sequence model. First released as a preprint (bioRxiv) in 2025.
- Melnikov A. *et al.* (2012) Systematic dissection and optimization of inducible enhancers in human cells using a massively parallel reporter assay. *Nat Biotechnol* 30:271–277.
- Arnold C. D., Gerlach D., Stelzer C., Boryń Ł. M., Rath M., Stark A. (2013) Genome-wide quantitative enhancer activity maps identified by STARR-seq. *Science* 339:1074–1077.
- Maurano M. T. *et al.* (2012) Systematic localization of common disease-associated variation in regulatory DNA. *Science* 337:1190–1195.
- ENCODE Project Consortium (2012) An integrated encyclopedia of DNA elements in the human genome. *Nature* 489:57–74.
- Castro-Mondragon J. A. *et al.* (2022) JASPAR 2022: the 9th release of the open-access database of transcription factor binding profiles. *Nucleic Acids Res* 50:D165–D173.
- Weirauch M. T. *et al.* (2014) Determination and inference of eukaryotic transcription factor sequence specificity. *Cell* 158:1431–1443.
- Schneider T. D., Stephens R. M. (1990) Sequence logos: a new way to display consensus sequences. *Nucleic Acids Res* 18:6097–6100.
- Durbin R., Eddy S., Krogh A., Mitchison G. (1998) *Biological Sequence Analysis*. Cambridge University Press (position weight matrices, Markov chains).

## ➡️ Next

DNA encodes proteins as well as their regulation. **[Module 37 · Protein sequences: from coevolution to protein
language models](37_protein_sequences_language_models.ipynb)** moves from the four-letter alphabet to twenty amino
acids — Potts models of coevolution, contacts, and protein language models that score mutations zero-shot, with the
same questions about what likelihood does and does not tell us.